# 第 14 集：Pandas 处理缺失数据

    > 对应《Numpy & Pandas 数据处理教程》课程。Notebook 按“概念 → 示例 → 观察结果”的顺序整理，建议逐格运行，并尝试修改示例数据。

    ## 本节目标


- 会识别和统计缺失值
- 掌握删除、填充、前后传播与插值
- 理解缺失值处理必须结合业务含义


## 1. 缺失值是什么

数值列常用 `NaN` 表示缺失。缺失不等于 0，也不等于空字符串；处理之前必须先判断它在当前业务中的含义。


In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame(
    {
        "温度": [20.0, np.nan, 24.0, 26.0],
        "湿度": [50.0, 55.0, np.nan, 65.0],
        "城市": ["北京", "上海", None, "广州"],
    },
    index=pd.date_range("2026-06-01", periods=4),
)
df


## 2. 检测并统计缺失值

`isna()` 逐格返回布尔值，继续 `.sum()` 可以统计每列缺失数量。`notna()` 则表示非缺失。


In [ ]:
missing_mask = df.isna()
missing_count = missing_mask.sum()

missing_mask, missing_count


## 3. 删除缺失数据

`dropna()` 默认删除含任意缺失值的行。可以用 `subset` 只检查关键列，也可用 `how="all"` 仅删除全为空的行。删除前要评估数据损失。


In [ ]:
complete_rows = df.dropna()
temperature_required = df.dropna(subset=["温度"])

complete_rows, temperature_required


## 4. 用固定值或统计量填充

固定值适合有明确默认含义的字段；均值或中位数适合某些数值分析，但会改变数据分布。下面对不同列采用不同策略。


In [ ]:
filled = df.fillna(
    {
        "温度": df["温度"].mean(),
        "湿度": df["湿度"].median(),
        "城市": "未知",
    }
)
filled


## 5. 前向填充、后向填充与插值

时间序列中，`ffill()` 使用上一个已知值，`bfill()` 使用下一个已知值，`interpolate()` 根据相邻数值估计。是否合理取决于数据变化规律。


In [ ]:
numeric = df[["温度", "湿度"]]
forward = numeric.ffill()
backward = numeric.bfill()
interpolated = numeric.interpolate()

forward, backward, interpolated


## 6. 处理后再次检查

缺失处理不应只运行一次函数就结束。再次统计缺失值，确认策略确实覆盖了目标列。


In [ ]:
assert filled.isna().sum().sum() == 0
filled.isna().sum()


## 本节小结

先统计缺失，再决定删除还是填充，最后复查。不存在适用于所有数据的填充值；“为什么缺失”比“调用哪个函数”更重要。
